In [ ]:
print("Customer Churn ML Notebook Started")

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

print("Snowflake session connected successfully")

In [ ]:
df = session.table("CUSTOMER_CHURN.ML_PIPE.TRAINING_DATA")

print("Table loaded successfully")
print("Rows:", df.count())
print("Columns:", len(df.columns))

In [ ]:
df.show(5)

In [ ]:


df.show(5)

In [ ]:
from snowflake.snowpark.functions import col, count, when

missing = df.select([
    count(when(col(c).is_null(), 1)).alias(c)
    for c in df.columns
])

missing.show()

In [ ]:
df_clean = df.filter(col("TOTALCHARGES").is_not_null())

print("Original rows:", df.count())
print("Cleaned rows:", df_clean.count())

In [ ]:
from snowflake.snowpark.functions import when, col

df_clean = df_clean.with_column(
    "CHURN",
    when(col("CHURN") == "Yes", 1).otherwise(0)
)

df_clean.select("CHURN").show(10)

In [ ]:
feature_columns = [
    "GENDER",
    "SENIORCITIZEN",
    "PARTNER",
    "DEPENDENTS",
    "TENURE",
    "PHONESERVICE",
    "MULTIPLELINES",
    "INTERNETSERVICE",
    "ONLINESECURITY",
    "ONLINEBACKUP",
    "DEVICEPROTECTION",
    "TECHSUPPORT",
    "STREAMINGTV",
    "STREAMINGMOVIES",
    "CONTRACT",
    "PAPERLESSBILLING",
    "PAYMENTMETHOD",
    "MONTHLYCHARGES",
    "TOTALCHARGES"
]

X = df_clean.select(feature_columns)
y = df_clean.select("CHURN")

print("Features:", len(feature_columns))
print("Rows:", df_clean.count())

In [ ]:
X_pd = X.to_pandas()
y_pd = y.to_pandas()["CHURN"]

print("X shape:", X_pd.shape)
print("y shape:", y_pd.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_columns = [
    "GENDER",
    "PARTNER",
    "DEPENDENTS",
    "PHONESERVICE",
    "MULTIPLELINES",
    "INTERNETSERVICE",
    "ONLINESECURITY",
    "ONLINEBACKUP",
    "DEVICEPROTECTION",
    "TECHSUPPORT",
    "STREAMINGTV",
    "STREAMINGMOVIES",
    "CONTRACT",
    "PAPERLESSBILLING",
    "PAYMENTMETHOD"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns)
    ],
    remainder="passthrough"
)

X_encoded = preprocessor.fit_transform(X_pd)

print("Encoded feature shape:", X_encoded.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y_pd,
    test_size=0.2,
    random_state=42,
    stratify=y_pd
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

logistic_model = LogisticRegression(max_iter=1000)

logistic_model.fit(X_train, y_train)

y_pred_logistic = logistic_model.predict(X_test)

accuracy_logistic = accuracy_score(y_test, y_pred_logistic)

print("Logistic Regression Accuracy:", accuracy_logistic)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_logistic))

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

logistic_pipeline = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("model", LogisticRegression(max_iter=3000))
])

logistic_pipeline.fit(X_train, y_train)

y_pred_logistic = logistic_pipeline.predict(X_test)

accuracy_logistic = accuracy_score(y_test, y_pred_logistic)

print("Logistic Regression Accuracy:", accuracy_logistic)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_logistic))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

random_forest_model.fit(X_train, y_train)

y_pred_rf = random_forest_model.predict(X_test)

accuracy_rf = accuracy_score(y_test, y_pred_rf)

print("Random Forest Accuracy:", accuracy_rf)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

comparison = {
    "Logistic Regression": [
        accuracy_score(y_test, y_pred_logistic),
        precision_score(y_test, y_pred_logistic),
        recall_score(y_test, y_pred_logistic),
        f1_score(y_test, y_pred_logistic)
    ],
    "Random Forest": [
        accuracy_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_rf)
    ]
}

comparison_df = pd.DataFrame(
    comparison,
    index=["Accuracy", "Precision", "Recall", "F1 Score"]
).T

comparison_df

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

comparison = {
    "Logistic Regression": [
        accuracy_score(y_test, y_pred_logistic),
        precision_score(y_test, y_pred_logistic),
        recall_score(y_test, y_pred_logistic),
        f1_score(y_test, y_pred_logistic)
    ],
    "Random Forest": [
        accuracy_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_rf)
    ]
}

comparison_df = pd.DataFrame(
    comparison,
    index=["Accuracy", "Precision", "Recall", "F1 Score"]
).T

comparison_df

In [ ]:
from sklearn.metrics import roc_auc_score

y_prob_logistic = logistic_pipeline.predict_proba(X_test)[:, 1]
y_prob_rf = random_forest_model.predict_proba(X_test)[:, 1]

auc_logistic = roc_auc_score(y_test, y_prob_logistic)
auc_rf = roc_auc_score(y_test, y_prob_rf)

print("Logistic Regression ROC-AUC:", auc_logistic)
print("Random Forest ROC-AUC:", auc_rf)

In [ ]:
from sklearn.metrics import confusion_matrix

cm_logistic = confusion_matrix(y_test, y_pred_logistic)

print("Logistic Regression Confusion Matrix:")
print(cm_logistic)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

fpr_logistic, tpr_logistic, _ = roc_curve(y_test, y_prob_logistic)

plt.figure(figsize=(7, 5))
plt.plot(
    fpr_logistic,
    tpr_logistic,
    label=f"Logistic Regression (AUC = {auc_logistic:.3f})"
)
plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Customer Churn Prediction")
plt.legend()
plt.show()

In [ ]:
comparison_df

In [ ]:
import joblib

joblib.dump(logistic_pipeline, "logistic_churn_model.pkl")
joblib.dump(preprocessor, "churn_preprocessor.pkl")

print("Model saved successfully!")
print("Preprocessor saved successfully!")

In [ ]:
new_customer = pd.DataFrame([{
    "GENDER": "Female",
    "SENIORCITIZEN": 0,
    "PARTNER": "Yes",
    "DEPENDENTS": "No",
    "TENURE": 5,
    "PHONESERVICE": "Yes",
    "MULTIPLELINES": "No",
    "INTERNETSERVICE": "Fiber optic",
    "ONLINESECURITY": "No",
    "ONLINEBACKUP": "No",
    "DEVICEPROTECTION": "No",
    "TECHSUPPORT": "No",
    "STREAMINGTV": "Yes",
    "STREAMINGMOVIES": "Yes",
    "CONTRACT": "Month-to-month",
    "PAPERLESSBILLING": "Yes",
    "PAYMENTMETHOD": "Electronic check",
    "MONTHLYCHARGES": 85.0,
    "TOTALCHARGES": 425.0
}])

new_customer_encoded = preprocessor.transform(new_customer)

prediction = logistic_pipeline.predict(new_customer_encoded)[0]
probability = logistic_pipeline.predict_proba(new_customer_encoded)[0][1]

print("Predicted Churn:", "Yes" if prediction == 1 else "No")
print("Churn Probability:", round(probability * 100, 2), "%")

In [ ]:
import os

print("logistic_churn_model.pkl exists:", os.path.exists("logistic_churn_model.pkl"))
print("churn_preprocessor.pkl exists:", os.path.exists("churn_preprocessor.pkl"))

In [ ]:
final_results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [accuracy_logistic, accuracy_rf],
    "Precision": [
        precision_score(y_test, y_pred_logistic),
        precision_score(y_test, y_pred_rf)
    ],
    "Recall": [
        recall_score(y_test, y_pred_logistic),
        recall_score(y_test, y_pred_rf)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_logistic),
        f1_score(y_test, y_pred_rf)
    ],
    "ROC-AUC": [auc_logistic, auc_rf]
})

final_results

In [ ]:
print("Selected Model: Logistic Regression")
print("Accuracy:", round(accuracy_logistic * 100, 2), "%")
print("ROC-AUC:", round(auc_logistic, 3))

In [ ]:

print("Intercept:", logistic_pipeline.named_steps["model"].intercept_[0])
print("Number of coefficients:", len(logistic_pipeline.named_steps["model"].coef_[0]))

print("Coefficients:")
print(logistic_pipeline.named_steps["model"].coef_[0])

In [ ]:
import numpy as np

np.set_printoptions(precision=17)

print("FEATURE NAMES:")
print(preprocessor.get_feature_names_out())

print("\nSCALER SCALES:")
print(logistic_pipeline.named_steps["scaler"].scale_)